# Fine-Tuning YOLOv8n for Object Detection

Fine-tunes a COCO-pretrained YOLOv8n model on a new dataset, evaluates it (precision, recall, mAP@50, mAP@50-95), measures speed (FPS), and saves the trained weights for live detection with `live_detect.py`.

**Run this on a GPU.** In Google Colab: *Runtime → Change runtime type → T4 GPU*.

Default dataset: Ultralytics **African Wildlife** (4 classes: buffalo, elephant, rhino, zebra), downloaded automatically.
To use your own labelled dataset instead (e.g. exported from Roboflow in YOLOv8 format), set `DATA` to its `data.yaml` path.

In [ ]:
!pip install -q -U ultralytics

In [ ]:
import torch, json, time
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU (training will be very slow)")

DATA = "african-wildlife.yaml"   # or "/content/my_dataset/data.yaml"
EPOCHS = 30
IMGSZ = 640

## 1. Train (fine-tune from COCO-pretrained weights)

In [ ]:
model = YOLO("yolov8n.pt")   # pretrained on COCO, used as the starting point
results = model.train(data=DATA, epochs=EPOCHS, imgsz=IMGSZ, batch=16, device=DEVICE,
                      seed=42, project="runs", name="finetune", exist_ok=True)

## 2. Evaluate on the validation and test splits

In [ ]:
best = YOLO("runs/finetune/weights/best.pt")

def box_metrics(split):
    m = best.val(data=DATA, split=split, imgsz=IMGSZ, device=DEVICE, plots=True)
    per_class = {best.names[i]: round(float(v), 3) for i, v in enumerate(m.box.maps)}
    return {
        "precision": round(float(m.box.mp), 3),
        "recall": round(float(m.box.mr), 3),
        "mAP50": round(float(m.box.map50), 3),
        "mAP50_95": round(float(m.box.map), 3),
        "per_class_mAP50_95": per_class,
    }

val_metrics = box_metrics("val")
try:
    test_metrics = box_metrics("test")
except Exception as e:
    print("No test split in this dataset:", e)
    test_metrics = None
print("VAL :", val_metrics)
print("TEST:", test_metrics)

## 3. Measure speed (FPS)
Average per-image time over 100 images, on GPU and on CPU. Report the device with the number.

In [ ]:
import glob, os
from ultralytics.data.utils import check_det_dataset

info = check_det_dataset(DATA)
img_dir = info.get("test") or info["val"]
images = sorted(glob.glob(os.path.join(img_dir, "*.*")))[:100]
print("Timing on", len(images), "images from", img_dir)

def measure(device):
    best.predict(images[0], device=device, verbose=False)   # warm-up
    times = []
    for p in images:
        r = best.predict(p, imgsz=IMGSZ, device=device, verbose=False)[0]
        times.append(sum(r.speed.values()))   # preprocess + inference + postprocess (ms)
    avg_ms = sum(times) / len(times)
    return {"avg_ms_per_image": round(avg_ms, 1), "fps": round(1000 / avg_ms, 1)}

speed = {"cpu": measure("cpu")}
if DEVICE == 0:
    speed["gpu"] = measure(0)
    speed["gpu_name"] = torch.cuda.get_device_name(0)
print(speed)

## 4. Sample predictions

In [ ]:
from IPython.display import Image, display

preds = best.predict(images[:6], imgsz=IMGSZ, device=DEVICE, conf=0.25,
                     save=True, project="runs", name="samples", exist_ok=True)
for p in sorted(glob.glob("runs/samples/*.jpg"))[:6]:
    display(Image(filename=p, width=500))

## 5. Save the metrics

In [ ]:
summary = {
    "base_model": "yolov8n.pt (COCO-pretrained)",
    "dataset": DATA,
    "classes": list(best.names.values()),
    "epochs": EPOCHS,
    "image_size": IMGSZ,
    "validation": val_metrics,
    "test": test_metrics,
    "speed": speed,
}
with open("yolo_metrics.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

## 6. Download the results
From the Colab file panel download:
- `runs/finetune/weights/best.pt` (trained model, use it with `live_detect.py`)
- `yolo_metrics.json`
- `runs/finetune/results.png`, `confusion_matrix.png`, `PR_curve.png`
- a few images from `runs/samples/`

Commit the metrics and plots (not the dataset) to GitHub, with this notebook's outputs saved.